In [4]:
import time
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
amap_API = os.getenv('amap_API')

In [6]:
df = pd.read_csv('MtF HRT Access Points.csv')

In [ ]:
def geocode_hospital(row):
    """Geocode a hospital name using the Amap API and return (lat, lng)."""
    hospital_name = row.get("Hospital")
    if pd.isna(hospital_name) or str(hospital_name).strip() == "":
        hospital_name = row.get("Chinese Hospital Name (if applicable)")

    city_name = row.get("City")

    if pd.isna(hospital_name) or str(hospital_name).strip() == "":
        return None, None

    address = str(hospital_name).strip()
    if pd.notna(city_name) and str(city_name).strip():
        address = f"{address}, {city_name}"

    url = "https://restapi.amap.com/v3/geocode/geo"
    params = {"key": amap_API, "address": address}
    if pd.notna(city_name) and str(city_name).strip():
        params["city"] = str(city_name).strip()

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("status") == "1" and data.get("geocodes"):
            location = data["geocodes"][0]["location"]
            lng, lat = map(float, location.split(","))
            return lat, lng
        else:
            print(f"No geocode returned for: {address}")
    except Exception as e:
        print(f"Geocoding failed for {address}: {e}")

    return None, None


In [ ]:
lats = []
lngs = []

for _, row in df.iterrows():
    lat, lng = geocode_hospital(row)
    lats.append(lat)
    lngs.append(lng)

# add to dataframe
df["latitude"] = lats
df["longitude"] = lngs

# save the file with coordinates
# df.to_csv("hospitals_with_coordinates.csv", index=False)


In [ ]:
df.to_csv("hospitals_with_coordinates.csv", index=False)
print(df[["Hospital", "Chinese Hospital Name (if applicable)", "latitude", "longitude"]].head())


In [ ]:
import json
import os

# Load the CSV that already contains coordinates, if present
if os.path.exists("hospitals_with_coordinates.csv"):
    df = pd.read_csv("hospitals_with_coordinates.csv")
else:
    df = pd.read_csv("MtF HRT Access Points.csv")

    if "latitude" not in df.columns or "longitude" not in df.columns:
        if "geocode_hospital" in globals():
            lats = []
            lngs = []
            for _, row in df.iterrows():
                lat, lng = geocode_hospital(row)
                lats.append(lat)
                lngs.append(lng)
            df["latitude"] = lats
            df["longitude"] = lngs
        else:
            raise NameError("Run the geocoding cell first so geocode_hospital is defined.")

# Keep the coordinate columns in the order longitude, latitude
if {"longitude", "latitude"}.issubset(df.columns):
    df = df.copy()
    df = df[[c for c in df.columns if c not in ["longitude", "latitude"]] + ["longitude", "latitude"]]

# Drop rows without coordinates
geo_df = df.dropna(subset=["longitude", "latitude"]).copy()


def clean_value(value):
    if pd.isna(value):
        return None
    return value

features = []
for _, row in geo_df.iterrows():
    properties = {
        "hospital": clean_value(row.get("Hospital")),
        "hospital_name": clean_value(row.get("Hospital")),
        "hospital_name_chinese": clean_value(row.get("Chinese Hospital Name (if applicable)")),
        "city": clean_value(row.get("City")),
        "province": clean_value(row.get("Province")),
        "department": clean_value(row.get("Department")),
        "can_continue": clean_value(row.get("Can Continue HRT")),
        "can_initial": clean_value(row.get("Can Initial HRT")),
        "can_guidance": clean_value(row.get("Can HRT Guidance"))
    }
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row["longitude"]), float(row["latitude"])]
        },
        "properties": properties
    })

geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open("hospitals_ready.geojson", "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2, allow_nan=False)

print(f"Wrote {len(features)} features to hospitals_ready.geojson")


Wrote 30 features to hospitals_ready.geojson
